# Funktionen für Parsen, Entitäten und Namespaces

heipy enthält mehrere Funktionen, die uns dabei helfen XML-Dateien mit lxml zu parsen und bearbeiten. Wir können diese Funktionen anhand der Datei lb-test.xml testen. Es handelt sich um eine einfache heiEditions-XML-Datei. Da werden Entites verwendet, die in https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/declarations/heieditions-entities.txt" definiert sind. Im nächsten Code-Abschnitt kann man den Inhalt der Datei sehen.

In [1]:
example_file_path = 'beispiele/beispiel_data/basic.xml'
import codecs
with codecs.open(example_file_path, 'r', 'utf-8') as example_file:
    example = example_file.read()
    print(example[:1000])

<?xml version='1.0' encoding='UTF-8'?>
<?xml-model href="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/tei_hes.rng" type="application/xml" schematypens="http://relaxng.org/ns/structure/1.0"?><?xml-model href="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/tei_hes.rng" type="application/xml" schematypens="http://purl.oclc.org/dsdl/schematron"?>
<!DOCTYPE TEI SYSTEM "https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS/declarations/heieditions-entities.txt">
<TEI xmlns="http://www.tei-c.org/ns/1.0" xmlns:hei="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS">
  <teiHeader>  
    <fileDesc>
      <titleStmt>
        <title ana="hc:MainTitle">Test</title>
      </titleStmt>
      <publicationStmt>
        <p>Very basic file for basic testing.</p>
      </publicationStmt>
      <sourceDesc>
        <p></p>
      </sourceDesc>
    </fileDesc>
  </teiHeader>
  <facsimile>
    <surface ana="hc:Page" n="1r" xml:id="_1r">
        <zone ana="hc:TextZone hc:Column"

Wenn wir so eine Datei mit lxml/etree parsen wollen, kommt eine Fehlermeldung, da die externe Entities nicht geladen werden können.

In [2]:
from lxml import etree as et

try:
    tree = et.parse(example_file_path)
except SyntaxError as e:
    print(e)
    pass

Entity 'bar' not defined, line 30, column 21 (basic.xml, line 30)


heipy bietet ein eigenes Parser (HeiEditionsParser), das diese Datei parsen kann. Alternativ kann man die Funktion heiparse() verwenden. Diese macht dazu noch automatisch ein **xinclude()**!

In [17]:
from heipy.parsers import HeiEditionsParser, heiparse

tree = et.parse(example_file_path, parser= HeiEditionsParser())
root = tree.getroot()
print(root)

tree = heiparse(example_file_path)
root = tree.getroot()
print(root)


<Element {http://www.tei-c.org/ns/1.0}TEI at 0x7863f815ce40>
<Element {http://www.tei-c.org/ns/1.0}TEI at 0x7863f8240180>


Wenn wir in einer TEI Datei XPath verwenden wollen, müssen wir entweder die Namespaces in geschweiften Klammern schreiben oder die Präfixe definieren. Also:

In [4]:
facsimile = root.findall('.//{http://www.tei-c.org/ns/1.0}facsimile')
facsimile = root.findall('.//tei:facsimile', namespaces= {'tei':'http://www.tei-c.org/ns/1.0'})

Die wichtigsten Präfixe (tei,xml,hei,hc,page,mets) werden in heipy schon in einer Variabel definiert, die importiert werden kann.

In [5]:
from heipy.namespaces import ns 

facsimile = root.findall('.//tei:facsimile', namespaces= ns)
print(facsimile)

[<Element {http://www.tei-c.org/ns/1.0}facsimile at 0x7863f82ed580>]


Manchmal müssen wir mit lxml auch diese Präfixe vor dem Elementname schreiben. Das können wir auch mit der Funktion `prefix_format` aus heipy machen. Hier ein Beispiel, wenn wir ein neues `<link>` Element in TEI-Namespace mit etree erzeugen wollen oder alle xml:id von `<l>` Elemente suchen.

In [6]:
from heipy.namespaces import prefix_format

new_elelement = et.Element(prefix_format('tei','link'))

for line in root.findall('.//tei:l', namespaces=ns):
    line_id = line.get(prefix_format('xml','id'))
    altn = line.get(prefix_format('hei','altN'))
    print(line_id, altn)

line_1_1 2


### Namespace-Klasse mit / Operator (Empfohlen)

Für eine elegantere Syntax kann man die `Namespace`-Klasse verwenden. Diese überlastet den `/` Operator, um Clark-Notation zu erzeugen. Die Namespace-Instanzen sind bereits vordefiniert und können direkt importiert werden:

In [ ]:
from heipy.namespaces import tei_ns, xml_ns, hei_ns

# Element-Tags in Clark-Notation erzeugen
lg_tag = tei_ns / "lg"
l_tag = tei_ns / "l"
link_tag = tei_ns / "link"

print(f"lg_tag: {lg_tag}")
print(f"l_tag: {l_tag}")

# Neues Element erzeugen
new_link = et.Element(tei_ns / "link")
print(f"\nNeues Element: {new_link}")

# Attribute setzen und lesen
for line in root.findall('.//tei:l', namespaces=ns):
    line_id = line.get(xml_ns / "id")
    altn = line.get(hei_ns / "altN")
    print(f"Line {line_id}, altN: {altn}")

**Verfügbare Namespace-Instanzen:**
- `tei_ns` - TEI namespace
- `xml_ns` - XML namespace  
- `hei_ns` - heiEDITIONS schema
- `hc_ns` - heiEditions ontology
- `page_ns`, `mets_ns`, `ex_ns`, `xi_ns` - weitere Namespaces

**Vergleich der Methoden:**
```python
# Alt: mit prefix_format()
element = et.Element(prefix_format('tei','link'))
id_val = line.get(prefix_format('xml','id'))

# Neu: mit / Operator (empfohlen)
element = et.Element(tei_ns / "link")
id_val = line.get(xml_ns / "id")
```

Die neue Methode ist kürzer, lesbarer und erfordert weniger Funktionsaufrufe.

# Transformations-Pipeline

Die Pipeline ist ein Submodul von heipy, das sich in `heipy.heipipe` befindet. Eine Pipeline besteht aus eine Serie von Schritten, die nacheinander ausgeführt werden. Es gibt unterschiedliche Arten von Schritten: XsltStep, PythonStep, AddAttribute, ValidationStep, DeleteStep. Diese Klassen sind in heipy.heipipe.steps definiert.

Schritte bekommen automatisch einen Name, wenn das `name` Parameter fehlt, aber es wird empfohlen, immer einen Name zu vergeben, damit man die Schritte besser identifizieren und die Pipelines besser bearbeiten kann.

Eine Pipeline wird aufgeführt mit der Funktion execute(), die eine XML Ausgangsdatei als Parameter verwendet.

In [7]:
from heipy.heipipe.steps import Pipeline, XsltStep, PythonStep, AddAttribute, ValidationStep, DeleteStep, UnwrapStep


pipe = Pipeline(name='Example_Pipe')

schritt1 = XsltStep(['src/heipy/heipipe/xslt/text_initials.xsl'], name="Initials")
pipe.add_step(schritt1)

schritt2 = XsltStep(['src/heipy/heipipe/xslt/text_markNoteAsEditorial.xsl'], name="Mark_note_as_editorial",
                    parameters= [{'note_classes': "hc:Comment"}])
pipe.add_step(schritt2)

pipe.add_step( DeleteStep(['facsimile']) )
# help(DeleteStep)

pipe.add_step(ValidationStep())
# help(ValidationStep)

pipe.add_step(AddAttribute('//tei:title', 'ana', 'hc:MainTitle'))
# help(AddAttribute)

pipe.add_step(UnwrapStep([{'element_name': 'w'}]))
# help(UnwrapStep)

for i, step in enumerate(pipe.get_steps()):
    print(f'{i}: {step}')

result = pipe.execute(example_file_path)



0: XSLStep »Initials« containing 1 transformations ['src/heipy/heipipe/xslt/text_initials.xsl'] and 0 parameters .
1: XSLStep »Mark_note_as_editorial« containing 1 transformations ['src/heipy/heipipe/xslt/text_markNoteAsEditorial.xsl'] and 1 parameters [{'note_classes': 'hc:Comment'}].
2: DeleteStep »__None__2«. Deletes: ['facsimile']
3: Validation step »validation«
4: Add Attribute Step »__None__4«. match: //tei:title, att_name: ana, att_val: hc:MainTitle
5: UnwrapStep »__None__5«. Unwraps: [{'element_name': 'w'}]
Starting Pipeline Example_Pipe for beispiele/beispiel_data/basic.xml
Validation succesful


Bei execute() kann man die parameter xinclude und egXML auf True setzen, wenn man diese Element richtig verarbeiten will.

A PythonStep is the most complex kind of step. It requires a function that takes as a parameter a root from an xml object element and a parameters argument that can be empty. It must return the edited root. For example:

In [8]:
def add_ptr_after_l(root, parameters):
    ls = root.findall('.//tei:l', namespaces=ns)
    for l in ls:
        et.SubElement(l, prefix_format('tei','ptr'), nsmap=ns)
    return root

pipe.add_step(PythonStep(add_ptr_after_l))

pipe.execute(example_file_path)


Starting Pipeline Example_Pipe for beispiele/beispiel_data/basic.xml
Validation succesful


'<TEI xmlns="http://www.tei-c.org/ns/1.0" xmlns:hei="https://digi.ub.uni-heidelberg.de/schema/tei/heiEDITIONS">\n  <teiHeader>  \n    <fileDesc>\n      <titleStmt>\n        <title ana="hc:MainTitle">Test</title>\n      </titleStmt>\n      <publicationStmt>\n        <p>Very basic file for basic testing.</p>\n      </publicationStmt>\n      <sourceDesc>\n        <p/>\n      </sourceDesc>\n    </fileDesc>\n  </teiHeader>\n  <facsimile>\n    <surface ana="hc:Page" n="1r" xml:id="_1r">\n        <zone ana="hc:TextZone hc:Column" n="1ra" xml:id="_1ra"/>\n    </surface>\n  </facsimile>\n  <text ana="hc:CompleteExpression">\n    <pb facs="#_1r"/>\n    <cb facs="#_1ra"/>\n    <body>\n      <lb n="1"/>\n      <l n="1" xml:id="line_1_1" hei:altN="2">\n        <w><hei:initial hei:color="Red" hei:heightLines="2">Z</hei:initial>u</w><c> </c>\n        <w>eine<g ref="char:bar">̄</g></w><c> </c>\n        <w>male</w><c> </c>\n        <w>mich</w><c> </c>\n        <w>ſere</w><c> </c>\n        <w>v<ex>er</e

Da die meisten Pipelines immer sehr ähnlich sind, werden default-Pipelines in heipy definiert. Zur Zeit gibt es semantic und sourceDoc

In [9]:
from heipy.heipipe.pipeline_library.sourcedoc import SourceDocPipe
from heipy.heipipe.pipeline_library.semantic import SemanticPipe

semantic_pipe = SemanticPipe()
sourcedoc_pipe = SourceDocPipe()

Die einzelnen Schritten der Default-Pipelines werden in heipy.heipipe.step_library definiert. Man kann diese mit dem get_steps() Funktion auflisten.

In [10]:
for i, step in enumerate(sourcedoc_pipe.get_steps()):
    print(f'{i}: {step}')

0: XSLStep »initials« containing 1 transformations ['text_initials.xsl'] and 0 parameters .
1: XSLStep »transcription_note« containing 1 transformations ['text_transcriptionNote.xsl'] and 0 parameters .
2: XSLStep »connect_lb_and_segment« containing 3 transformations ['text_connectLbWithZone.xsl', 'text_moveIncludedInZone.xsl', 'text_connectSegmentWithLine.xsl'] and 0 parameters .
3: Python step »move_physical_beginnings«, using: <function move_physical_beginnings at 0x7863f8164180>
4: XSLStep »Whitespaces« containing 4 transformations ['text_trimWhitespaceAdjacentToPhysicalBeginnings.xsl', 'text_normalizeWhitespaceInMixedContent.xsl', 'text_stripWhitespaceInElementsStatedBySchema.xsl', 'text_normalizeWhitespaceInTokenizedContent.xsl'] and 0 parameters .
5: XSLStep »mark_note_as_editorial« containing 1 transformations ['text_markNoteAsEditorial.xsl'] and 1 parameters [{'note_classes': 'hc:TextCriticalNote hc:TranscriptionNote hc:TextConstitutionNote hc:Comment hc:FontesNote hc:VariantN

Man kann neue Schritte hinzufügen mit add_step(). Per Default am Ende der Pipeline, aber mit möglich index (s. oben um die Indexes zu sehen). Alternativ kann man after_step() oder before_step() verwenden. Man kann auch Schritte löschen, entweder nach Schrittname (step_name) oder nach Index (step_index).

In [11]:
sourcedoc_pipe = SourceDocPipe()

sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], 
                                 name="new_step_1"))

sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], 
                                 name="new_step_2"), 
                                 at_index= 0)

sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl'], 
                                 name="new_step_3"), 
                                 after_step='Whitespaces')

sourcedoc_pipe.add_step(XsltStep(files=['pipelines/local_transformations/editorial_pc.xsl']), 
                                 before_step='number_line_segment_beginnings')

sourcedoc_pipe.set_pipestep_parameter('mark_note_as_editorial','note_classes', 'hc:Comment')

sourcedoc_pipe.remove_step(step_name='initials')

removed2 = sourcedoc_pipe.remove_step(step_index=5) # You can store the removed step in a variable if desired

for i, step in enumerate(sourcedoc_pipe.get_steps()):
    print(f'{i}: {step}')


0: XSLStep »new_step_2« containing 1 transformations ['pipelines/local_transformations/editorial_pc.xsl'] and 0 parameters .
1: XSLStep »transcription_note« containing 1 transformations ['text_transcriptionNote.xsl'] and 0 parameters .
2: XSLStep »connect_lb_and_segment« containing 3 transformations ['text_connectLbWithZone.xsl', 'text_moveIncludedInZone.xsl', 'text_connectSegmentWithLine.xsl'] and 0 parameters .
3: Python step »move_physical_beginnings«, using: <function move_physical_beginnings at 0x7863f8164180>
4: XSLStep »Whitespaces« containing 4 transformations ['text_trimWhitespaceAdjacentToPhysicalBeginnings.xsl', 'text_normalizeWhitespaceInMixedContent.xsl', 'text_stripWhitespaceInElementsStatedBySchema.xsl', 'text_normalizeWhitespaceInTokenizedContent.xsl'] and 0 parameters .
5: XSLStep »mark_note_as_editorial« containing 1 transformations ['text_markNoteAsEditorial.xsl'] and 1 parameters [{'note_classes': 'hc:Comment'}].
6: XSLStep »__None__8« containing 1 transformations [

Die Schritte der Default-Pipelines werden in `heipy.heipipe.step_library` definiert. Als Beispiel hier `heipy.heipipe.step_library.ptr2ref`

In [12]:
# from ..steps import XsltStep

def get_step():
    return XsltStep(files=[
    "ptr2Ref.xsl",
    ], name="ptr2ref", pipe_files=True)

"ptr2Ref.xsl" ist eine Datei in `heipy.heipipe.xslt` 
Die Dateien in diesem Verzeichnis können ohne Path-Angabe direkt im Parameter 'files' angegeben werden mit dem Parameter `pipe_files= True`. 

# Weitere Funktionen

Alle Charaktere in einem String auflisten

In [13]:
from heipy.validation import list_all_characters

example = "This is an exāple!"

list_all_characters(example)


[(' ', 'U+0020'),
 ('!', 'U+0021'),
 ('T', 'U+0054'),
 ('a', 'U+0061'),
 ('e', 'U+0065'),
 ('h', 'U+0068'),
 ('i', 'U+0069'),
 ('l', 'U+006C'),
 ('n', 'U+006E'),
 ('p', 'U+0070'),
 ('s', 'U+0073'),
 ('x', 'U+0078'),
 ('̄', 'U+0304')]